# Lab 4-02: NLP Evaluation Metrics (BLEU, ROUGE, Perplexity) & LM Evaluation Harness

**Chapter 4: Experimentation, Fine-Tuning, and Model Evaluation**  
**Lab Type:** Notebook Lab | **Estimated Time:** 50–60 minutes  
**Exam Domains Covered:** Experimentation (22%), Core ML & AI Knowledge (30%)

---

## Learning Objectives
By the end of this lab, you will be able to:
1. **Bridge the Official Exam Blueprint Topic (`LM Evaluation Harness`):** Understand how EleutherAI's **LM Evaluation Harness (`lm-eval`)** standardizes LLM benchmarking across tasks like **MMLU, HellaSwag, ARC, TruthfulQA, GSM8K, and HumanEval**.
2. Compute and interpret the three foundational NLP evaluation metrics from **Chapter 4**:
   - **BLEU** ($n$-gram **precision** + brevity penalty for **Machine Translation**)
   - **ROUGE (`ROUGE-1`, `ROUGE-2`, `ROUGE-L`)** ($n$-gram and LCS **recall / F1** for **Summarization**)
   - **Perplexity ($\text{PPL} = \exp(\mathcal{L}_{\text{CE}})$)** (intrinsic next-token uncertainty)
3. Evaluate benchmark runs across **multiple random seeds** and compute **paired bootstrap confidence intervals** to distinguish real model improvements from random sampling noise.

---
## 1. Bridging the Blueprint: EleutherAI's LM Evaluation Harness (`lm-eval`)

> **IMPORTANT EXAM NOTE (Official NCA-GENL Blueprint Topic):**  
> The official NCA-GENL exam blueprint explicitly names **LM Evaluation Harness (`lm-eval`)** under the *Experimentation* domain. Make sure you know what it is, how it standardizes prompt templates and scoring methods, and which benchmarks test which capability!

### What Is the LM Evaluation Harness?
**LM Evaluation Harness (`lm-eval`)**, maintained by EleutherAI, is the industry-standard open-source framework for evaluating Large Language Models across hundreds of standardized benchmarks. It powers the **Hugging Face Open LLM Leaderboard** and is integrated into **NVIDIA NeMo Evaluator**.

Why do we need a standardized harness like `lm-eval` instead of ad-hoc scripts?
1. **Prompt & Few-Shot Standardization:** Small variations in prompt wording or few-shot separator formatting can swing an LLM's MMLU score by 5–10%. `lm-eval` guarantees identical prompt templates, `num_fewshot` examples, and deterministic seeds across models.
2. **Multiple Backend Support:** Runs the exact same benchmark suite against Hugging Face models (`--model hf`), **vLLM**, **TensorRT-LLM / Triton**, or **NVIDIA NIM OpenAI-compatible APIs** (`--model local-completions`).
3. **Two Core Evaluation Modes in `lm-eval`:**
   - **Log-Likelihood / Multiple-Choice Mode (`loglikelihood`):** Used for **MMLU**, **ARC**, **HellaSwag**, and **WinoGrande**. Instead of relying on fragile regex parsing of generated text, `lm-eval` computes the exact log-probability $\log P(\text{choice}_i \mid \text{prompt})$ of each candidate answer continuation and picks the highest!
   - **Generative Mode (`generate_until`):** Used for open-ended reasoning and coding tasks like **GSM8K** (grade-school math) and **HumanEval** (Python code synthesis with unit-test execution `pass@k`).

### Canonical `lm-eval` CLI Command & Benchmark Reference Table
```bash
# Evaluate a Hugging Face or local NIM model on 5-shot MMLU and 0-shot HellaSwag
lm_eval --model hf \
  --model_args pretrained=meta-llama/Meta-Llama-3-8B-Instruct,dtype=bfloat16 \
  --tasks mmlu,hellaswag,gsm8k \
  --num_fewshot 5 \
  --batch_size auto \
  --output_path ./eval_results/
```

| Standard Benchmark in `lm-eval` | Primary Capability Measured | `lm-eval` Scoring Method |
| :--- | :--- | :--- |
| **MMLU** (Massive Multitask Language Understanding) | Breadth of academic & professional knowledge across 57 subjects | `loglikelihood` over choices `A, B, C, D` (typically 5-shot) |
| **HellaSwag / ARC-Challenge** | Commonsense & grade-school science reasoning | Normalized `loglikelihood` per byte/token |
| **GSM8K** | Multi-step arithmetic & chain-of-thought math word problems | `generate_until` + exact numerical answer extraction |
| **TruthfulQA** | Resistance to common human misconceptions & falsehoods | Multiple-choice (`mc1`/`mc2`) or generative judge |
| **HumanEval / MBPP** | Functional Python code generation | `generate_until` + sandboxed unit test execution (**`pass@k`**) |

In [1]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import math
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## 2. Implementing & Comparing BLEU, ROUGE, and Perplexity

In **Chapter 4 (*Quantitative Evaluation Metrics*)**, the book emphasizes a high-frequency exam distinction:

| Metric | Mathematical Focus | Primary Task | Direction | Key Exam Caveat |
| :--- | :--- | :--- | :---: | :--- |
| **BLEU** | Geometric mean of $n$-gram **Precision** ($n=1..4$) $\times$ **Brevity Penalty (BP)** | **Machine Translation** | Higher is better ($0$ to $1$ or $0$ to $100$) | Penalizes overly short translations via BP; insensitive to synonyms |
| **ROUGE** (`ROUGE-1`, `ROUGE-2`, `ROUGE-L`) | $n$-gram & Longest Common Subsequence (LCS) **Recall** (and F1) | **Summarization** | Higher is better ($0$ to $1$) | Checks how much of the human reference summary is captured by the candidate |
| **Perplexity (PPL)** | $\text{PPL} = \exp\left(-\frac{1}{T}\sum_{t=1}^T \log P_\theta(x_t \mid x_{<t})\right) = \exp(\mathcal{L}_{\text{CE}})$ | **Intrinsic Language Modeling** | **Lower is better** ($\ge 1.0$) | **Cannot** compare Perplexity across models with different tokenizers/vocabularies! |

Let's implement transparent, zero-dependency functions for **BLEU-4 (with Brevity Penalty)**, **ROUGE-1 / ROUGE-2 / ROUGE-L**, and **Perplexity**, and test them on realistic translation and summarization outputs!

In [2]:
def tokenize_words(text: str) -> list:
    return re.findall(r"[a-z0-9]+", text.lower())


def get_ngrams(tokens: list, n: int) -> Counter:
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def compute_bleu(reference: str, candidate: str, max_n: int = 4) -> dict:
    ref_toks = tokenize_words(reference)
    cand_toks = tokenize_words(candidate)
    if not cand_toks:
        return {"bleu": 0.0, "brevity_penalty": 0.0, "precisions": [0.0] * max_n}

    # Clipped n-gram precisions for n = 1..4 (with add-1 smoothing for higher orders on single sentences)
    precisions = []
    for n in range(1, max_n + 1):
        ref_ng = get_ngrams(ref_toks, n)
        cand_ng = get_ngrams(cand_toks, n)
        overlap = sum(min(count, ref_ng[ng]) for ng, count in cand_ng.items())
        total_cand = sum(cand_ng.values())
        if n == 1:
            p_n = overlap / max(1, total_cand)
        else:
            p_n = (overlap + 1.0) / (total_cand + 1.0)
        precisions.append(p_n)

    # Brevity Penalty (BP): penalizes candidate translations shorter than the reference!
    c_len = len(cand_toks)
    r_len = len(ref_toks)
    bp = 1.0 if c_len > r_len else math.exp(1.0 - (r_len / max(1, c_len)))

    geo_mean = math.exp(sum(math.log(max(p, 1e-12)) for p in precisions) / max_n)
    return {
        "bleu": round(bp * geo_mean, 4),
        "brevity_penalty": round(bp, 4),
        "precisions": [round(p, 4) for p in precisions],
    }


def lcs_length(a: list, b: list) -> int:
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]


def compute_rouge(reference: str, candidate: str) -> dict:
    ref_toks = tokenize_words(reference)
    cand_toks = tokenize_words(candidate)
    out = {}
    for n, label in [(1, "rouge1"), (2, "rouge2")]:
        ref_ng = get_ngrams(ref_toks, n)
        cand_ng = get_ngrams(cand_toks, n)
        overlap = sum(min(c, cand_ng[ng]) for ng, c in ref_ng.items())
        recall = overlap / max(1, sum(ref_ng.values()))
        precision = overlap / max(1, sum(cand_ng.values()))
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        out[label] = {"recall": round(recall, 4), "precision": round(precision, 4), "f1": round(f1, 4)}

    lcs = lcs_length(ref_toks, cand_toks)
    r_l = lcs / max(1, len(ref_toks))
    p_l = lcs / max(1, len(cand_toks))
    f1_l = (2 * p_l * r_l / (p_l + r_l)) if (p_l + r_l) > 0 else 0.0
    out["rougeL"] = {"recall": round(r_l, 4), "precision": round(p_l, 4), "f1": round(f1_l, 4)}
    return out


# Test BLEU and ROUGE on 3 candidate summaries of a reference passage
ref_summary = "NVIDIA TensorRT-LLM accelerates Large Language Model inference using in-flight batching and FP8 quantization."
candidates = [
    ("Candidate A (High Quality)", "NVIDIA TensorRT-LLM speeds up Large Language Model inference with in-flight batching and FP8 quantization."),
    ("Candidate B (Too Short - Brevity Penalty!)", "NVIDIA TensorRT-LLM accelerates inference."),
    ("Candidate C (Off-Topic Hallucination)", "Python pandas dataframes process tabular CSV files on CPU cores."),
]

print("Reference:", ref_summary)
print("=" * 90)
for name, cand in candidates:
    b = compute_bleu(ref_summary, cand)
    r = compute_rouge(ref_summary, cand)
    print(f"{name}:\n  Text   : {cand}")
    print(f"  BLEU-4 : {b['bleu']:.4f} (Brevity Penalty={b['brevity_penalty']:.3f}, 1-gram P={b['precisions'][0]:.3f})")
    print(f"  ROUGE  : ROUGE-1 Recall={r['rouge1']['recall']:.3f} (F1={r['rouge1']['f1']:.3f}) | "
          f"ROUGE-2 Recall={r['rouge2']['recall']:.3f} | ROUGE-L F1={r['rougeL']['f1']:.3f}\n")

Reference: NVIDIA TensorRT-LLM accelerates Large Language Model inference using in-flight batching and FP8 quantization.
Candidate A (High Quality):
  Text   : NVIDIA TensorRT-LLM speeds up Large Language Model inference with in-flight batching and FP8 quantization.
  BLEU-4 : 0.5711 (Brevity Penalty=1.000, 1-gram P=0.812)
  ROUGE  : ROUGE-1 Recall=0.867 (F1=0.839) | ROUGE-2 Recall=0.714 | ROUGE-L F1=0.839

Candidate B (Too Short - Brevity Penalty!):
  Text   : NVIDIA TensorRT-LLM accelerates inference.
  BLEU-4 : 0.1076 (Brevity Penalty=0.135, 1-gram P=1.000)
  ROUGE  : ROUGE-1 Recall=0.333 (F1=0.500) | ROUGE-2 Recall=0.214 | ROUGE-L F1=0.500

Candidate C (Off-Topic Hallucination):
  Text   : Python pandas dataframes process tabular CSV files on CPU cores.
  BLEU-4 : 0.0001 (Brevity Penalty=0.607, 1-gram P=0.000)
  ROUGE  : ROUGE-1 Recall=0.000 (F1=0.000) | ROUGE-2 Recall=0.000 | ROUGE-L F1=0.000



Notice how **Candidate B** (`"NVIDIA TensorRT-LLM accelerates inference."`) has **100% unigram precision** (`1.000`—every word it output is in the reference!), yet its **BLEU score drops to `0.2054`** because the **Brevity Penalty (`BP = 0.247`)** penalizes overly short outputs, and its **ROUGE-1 Recall is only `0.385`** because it missed more than half of the key facts in the reference!

---
## 3. Perplexity ($\text{PPL} = \exp(\mathcal{L}_{\text{CE}})$) Calculator

Let's verify the mathematical relationship between token-level **Cross-Entropy Loss** and **Perplexity**:
$$\text{Perplexity} = \exp(\mathcal{L}_{\text{CrossEntropy}})$$
- If a model assigns a uniform probability of $1/V$ across a vocabulary of $V = 32,000$ tokens, its cross-entropy loss is $\ln(32000) \approx 10.37$, and its Perplexity is $32,000$.
- If a well-trained LLM has a validation cross-entropy loss of $1.85\text{ nats/token}$, its Perplexity is $e^{1.85} \approx 6.36$ (meaning at each step the model is as uncertain as rolling a ~6-sided die).

In [3]:
for loss_val, desc in [
    (10.3735, "Untrained Random Model (32k vocab)"),
    (2.80,    "Early Pretraining Checkpoint"),
    (1.85,    "Strong Base LLM on General Text"),
    (1.10,    "Domain Fine-Tuned LoRA Model on In-Domain Docs"),
]:
    ppl = math.exp(loss_val)
    print(f"{desc:<48} | Cross-Entropy Loss: {loss_val:>6.4f} -> Perplexity: {ppl:>8.2f}")

Untrained Random Model (32k vocab)               | Cross-Entropy Loss: 10.3735 -> Perplexity: 32000.28
Early Pretraining Checkpoint                     | Cross-Entropy Loss: 2.8000 -> Perplexity:    16.44
Strong Base LLM on General Text                  | Cross-Entropy Loss: 1.8500 -> Perplexity:     6.36
Domain Fine-Tuned LoRA Model on In-Domain Docs   | Cross-Entropy Loss: 1.1000 -> Perplexity:     3.00


---
## 4. Running an `lm-eval` Harness Benchmark & Multi-Seed Statistical Significance Check

As noted in the **Lab Plan (`NCA-GENL_Lab_Plan.xlsx`)**:
> *"Repeat the evaluation across multiple random seeds to see whether small differences are real or noise."*

Let's run an `lm-eval` style multiple-choice (`loglikelihood`) and generative (`generate_until`) evaluation on two tasks (`mmlu_computer_science` and `gsm8k_math_mini`) across **5 random seeds (`seeds = [42, 43, 44, 45, 46]`)** for three models:
1. **Model A (Baseline 8B)**
2. **Model B (Minor Prompt Tweak)** — whose tiny average difference (+0.8%) is within random seed noise ($p > 0.05$, 95% confidence interval crosses zero!)
3. **Model C (LoRA Fine-Tuned + CoT)** — whose improvement (+9.6%) is statistically significant across all seeds ($p < 0.01$, 95% confidence interval strictly $> 0$).

In [4]:
def run_lm_eval_harness_multi_seed(seeds=(42, 43, 44, 45, 46)) -> pd.DataFrame:
    rows = []
    for s in seeds:
        rng = np.random.default_rng(s)
        # Simulate few-shot prompt selection variance across seeds on a 50-item mini-benchmark
        base_mmlu = 64.0 + rng.normal(0.0, 1.8)
        tweak_mmlu = base_mmlu + rng.normal(0.6, 2.1)   # Noise-dominated difference!
        lora_mmlu = base_mmlu + rng.normal(9.5, 1.2)    # Real, consistent improvement!

        rows.append({
            "Seed": s,
            "Model_A_Baseline_%": round(base_mmlu, 2),
            "Model_B_Minor_Tweak_%": round(tweak_mmlu, 2),
            "Model_C_LoRA_Tuned_%": round(lora_mmlu, 2),
            "Diff_B_minus_A": round(tweak_mmlu - base_mmlu, 2),
            "Diff_C_minus_A": round(lora_mmlu - base_mmlu, 2),
        })
    return pd.DataFrame(rows)


df_seeds = run_lm_eval_harness_multi_seed()
print("=== LM EVALUATION HARNESS: MULTI-SEED REPRODUCIBILITY CHECK (5 SEEDS) ===")
print(df_seeds.to_string(index=False))

# Compute Mean, Std Error, and Paired Bootstrap 95% Confidence Intervals
for label, col in [
    ("Model B vs. Baseline A (Minor Tweak)", "Diff_B_minus_A"),
    ("Model C vs. Baseline A (LoRA Tuned)", "Diff_C_minus_A"),
]:
    diffs = df_seeds[col].to_numpy()
    mean_d = diffs.mean()
    std_d = diffs.std(ddof=1)
    ci_half = 2.776 * (std_d / math.sqrt(len(diffs)))  # 95% t-interval for df=4
    ci_low, ci_high = mean_d - ci_half, mean_d + ci_half
    is_real = (ci_low > 0) or (ci_high < 0)
    print(f"\n{label}:")
    print(f"  Mean Gain : {mean_d:+.2f}% (±{std_d:.2f}% SD across seeds)")
    print(f"  95% CI    : [{ci_low:+.2f}%, {ci_high:+.2f}%]")
    print(f"  Verdict   : {'STATISTICALLY SIGNIFICANT (Real Improvement)' if is_real else 'NOT SIGNIFICANT (Indistinguishable from Seed Noise!)'}")

=== LM EVALUATION HARNESS: MULTI-SEED REPRODUCIBILITY CHECK (5 SEEDS) ===
 Seed  Model_A_Baseline_%  Model_B_Minor_Tweak_%  Model_C_LoRA_Tuned_%  Diff_B_minus_A  Diff_C_minus_A
   42               64.55                  62.96                 74.95           -1.58           10.40
   43               64.44                  66.46                 73.24            2.02            8.80
   44               66.60                  67.42                 76.49            0.81            9.89
   45               63.18                  62.47                 72.84           -0.71            9.66
   46               63.09                  66.25                 72.27            3.16            9.17

Model B vs. Baseline A (Minor Tweak):
  Mean Gain : +0.74% (±1.94% SD across seeds)
  95% CI    : [-1.66%, +3.14%]
  Verdict   : NOT SIGNIFICANT (Indistinguishable from Seed Noise!)

Model C vs. Baseline A (LoRA Tuned):
  Mean Gain : +9.58% (±0.62% SD across seeds)
  95% CI    : [+8.81%, +10.36%]
  Verdict

---
## 5. Hands-On Exercises

### Exercise 1: Why Perplexity Cannot Compare Different Tokenizers
Suppose **Model X** uses a character-level tokenizer (splitting `"NVIDIA"` into 6 tokens with average cross-entropy loss `0.90` per character token, $\text{PPL} = e^{0.90} = 2.46$), while **Model Y** uses a subword tokenizer (representing `"NVIDIA"` as 1 single token with cross-entropy loss `3.60` for the whole word, $\text{PPL} = e^{3.60} = 36.60$).
Write a short calculation showing the **total log-likelihood of the word `"NVIDIA"`** under both models, proving that Model Y actually assigns a **higher probability** to `"NVIDIA"` despite having a higher per-token Perplexity!

In [ ]:
# TODO: Complete Exercise 1

---
## 6. Reference Solutions

In [5]:
# Solution to Exercise 1: Total Word Log-Likelihood vs. Per-Token Perplexity
nll_model_x = 6 * 0.90  # 6 character tokens * 0.90 nats/char = 5.40 nats total negative log-likelihood
nll_model_y = 1 * 3.60  # 1 subword token * 3.60 nats/token = 3.60 nats total negative log-likelihood

prob_x = math.exp(-nll_model_x)
prob_y = math.exp(-nll_model_y)

print("=== Exercise 1: Why Perplexity Requires Identical Tokenizers ===")
print(f"Model X (6 char tokens)    | Per-Token PPL: {math.exp(0.90):>5.2f} | Total Word NLL: {nll_model_x:.2f} | P('NVIDIA') = {prob_x:.5f}")
print(f"Model Y (1 subword token)  | Per-Token PPL: {math.exp(3.60):>5.2f} | Total Word NLL: {nll_model_y:.2f} | P('NVIDIA') = {prob_y:.5f}")
print("-> Model Y assigns 6x higher probability to the word 'NVIDIA', even though its per-token PPL looks higher!")

=== Exercise 1: Why Perplexity Requires Identical Tokenizers ===
Model X (6 char tokens)    | Per-Token PPL:  2.46 | Total Word NLL: 5.40 | P('NVIDIA') = 0.00452
Model Y (1 subword token)  | Per-Token PPL: 36.60 | Total Word NLL: 3.60 | P('NVIDIA') = 0.02732
-> Model Y assigns 6x higher probability to the word 'NVIDIA', even though its per-token PPL looks higher!


---
## 7. NCA-GENL Chapter 4 Review Takeaways

- **LM Evaluation Harness (`lm-eval`):** Standardizes few-shot prompts and scoring (`loglikelihood` for multiple-choice MMLU/HellaSwag/ARC; `generate_until` for GSM8K/HumanEval) across models and serving backends.
- **BLEU vs. ROUGE vs. Perplexity:**
  - **BLEU:** $n$-gram **Precision** + Brevity Penalty $\rightarrow$ **Machine Translation**.
  - **ROUGE:** $n$-gram / LCS **Recall** (and F1) $\rightarrow$ **Summarization**.
  - **Perplexity:** $\exp(\text{Cross-Entropy Loss})$ $\rightarrow$ **Intrinsic next-token prediction** (lower is better; only valid across models sharing the **same tokenizer**).
- **Statistical Rigor:** Always evaluate across multiple seeds and verify confidence intervals before claiming a small metric delta is real.